# Bloom's Taxonomy Chunk Tagging

**Required Kaggle datasets (add via 'Add Data'):**
- `bloom-chunks` → contains `chunks.json`
- `bloom-bert-classifier` → contains `checkpoint-600/` folder

**Accelerator:** GPU T4 x2

**Output:** `/kaggle/working/chunks_tagged.json` — download and replace `data/chunks.json` locally,
then run `python scripts/export_for_web.py`.

In [ ]:
import subprocess, sys, os

result = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
cuda_ver = ""
for line in result.stdout.splitlines():
    if "release" in line.lower():
        cuda_ver = line.strip()
        break
print(f"CUDA: {cuda_ver or 'not found'}")

BAD_CUDA     = "13.2" in cuda_ver
HAS_CUDA     = bool(cuda_ver) and not BAD_CUDA
N_GPU_LAYERS = -1 if HAS_CUDA else 0
print(f"GPU offload: {'YES' if HAS_CUDA else 'NO'}")

# Try wheels newest → oldest; fall back to CPU build if all fail
CUDA_TAGS = ["cu124", "cu123", "cu122", "cu121"] if HAS_CUDA else []
installed = False
for tag in CUDA_TAGS:
    url = f"https://abetlen.github.io/llama-cpp-python/whl/{tag}"
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "llama-cpp-python", "-q",
         "--extra-index-url", url],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print(f"Installed llama-cpp-python ({tag})")
        installed = True
        break
    print(f"  {tag} failed, trying next...")


In [ ]:
from llama_cpp import Llama, llama_supports_gpu_offload
print("GPU offload supported:", llama_supports_gpu_offload())

# Quick smoke test — load a tiny model or just check CUDA context
import ctypes
try:
    ctypes.CDLL("libcudart.so")
    print("libcudart: OK")
except Exception as e:
    print(f"libcudart: FAILED — {e}")


In [ ]:
import builtins
builtins.__NOTEBOOK__ = True

In [ ]:
"""
Gemma 4 learning-goal extractor for Bloom's taxonomy tagging.

Downloads the GGUF model on first run (cached by llama_cpp / HuggingFace Hub).
Subsequent runs load from the local cache — no network required.

Usage — import:
    from gemma_goal_extractor import load_gemma, extract_goals

    llm   = load_gemma()
    goals = extract_goals(chunk_text, llm)
    # goals → ["Understand the concept of X", "Apply Y to solve Z", ...]

Usage — CLI (test a single chunk):
    python scripts/gemma_goal_extractor.py --text "Some chunk text here..."
    python scripts/gemma_goal_extractor.py --file path/to/chunk.txt

Flags:
    --text TEXT       Chunk text to extract goals from (quoted string)
    --file FILE       Path to a plain-text file containing the chunk
    --repo-id ID      HuggingFace repo (default: unsloth/gemma-4-E4B-it-GGUF)
    --filename FILE   GGUF filename    (default: gemma-4-E4B-it-Q4_K_M.gguf)
    --n-ctx N         Context window size (default: 8192)
    --max-goals N     Maximum goals to extract per chunk (default: 10)
"""

# Fix for notebook environment — __file__ is not defined in cells
from pathlib import Path
__file__ = "/kaggle/working/gemma_goal_extractor.py"


from __future__ import annotations

import re
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# Prompt
# ---------------------------------------------------------------------------

_SYSTEM_PROMPT = (
    "You are an educational content analyst. Your only job is to extract "
    "learning goals or outcomes from academic text."
)

_USER_TEMPLATE = """\
Read the following text and extract the key learning goals or outcomes it \
addresses — what a student would learn, understand, apply, analyze, evaluate, \
or be able to create after studying this material.

Return ONLY a numbered list of concise learning outcome sentences (one per \
line, no headings, no extra explanation). Each sentence should start with a \
Bloom's verb (e.g. "Remember...", "Evaluate ...", "Create ..." etc). \
If the text contains no clear learning goals, return exactly: NONE

Text:
{chunk_text}
"""

# ---------------------------------------------------------------------------
# Model loading
# ---------------------------------------------------------------------------

_PROJECT_ROOT = Path(__file__).resolve().parent.parent
_IS_KAGGLE    = Path('/kaggle/input').exists()
_MODELS_DIR   = (
    Path('/kaggle/working/models') if _IS_KAGGLE
    else _PROJECT_ROOT / "webapp" / "models"
)


def load_gemma(
    repo_id: str      = "unsloth/gemma-4-E4B-it-GGUF",
    filename: str     = "gemma-4-E4B-it-Q4_K_M.gguf",
    n_ctx: int        = 8192,
    n_gpu_layers: int = -1,
):
    """Load Gemma 4 GGUF via llama_cpp. Downloads to webapp/models/ (or /kaggle/working/models/) on first run.

    n_gpu_layers=-1 offloads all layers to GPU when available; falls back to CPU automatically.
    """
    try:
        from llama_cpp import Llama
    except ImportError:
        print("Error: llama_cpp not installed. Run: pip install llama-cpp-python")
        sys.exit(1)

    local_path = _MODELS_DIR / filename
    _MODELS_DIR.mkdir(parents=True, exist_ok=True)

    if local_path.exists():
        print(f"[Gemma] Loading from local cache: {local_path}")
        llm = Llama(model_path=str(local_path), n_ctx=n_ctx,
                    n_gpu_layers=n_gpu_layers, verbose=False)
    else:
        print(f"[Gemma] Downloading {filename} → {_MODELS_DIR} ...")
        llm = Llama.from_pretrained(
            repo_id=repo_id,
            filename=filename,
            n_ctx=n_ctx,
            n_gpu_layers=n_gpu_layers,
            verbose=False,
            local_dir=str(_MODELS_DIR),
        )
        print(f"[Gemma] Saved to {local_path}")
    print("[Gemma] Model ready.")
    return llm


# ---------------------------------------------------------------------------
# Goal extraction
# ---------------------------------------------------------------------------

def extract_goals(
    chunk_text: str,
    llm,
) -> list[str]:
    """
    Extract learning goals/outcomes from a chunk using Gemma 4.

    Returns a list of short goal strings (empty list if none found or on error).
    Each goal is suitable as direct input to the BERT Bloom's classifier.
    """
    prompt = _USER_TEMPLATE.format(chunk_text=chunk_text.strip())
    try:
        response = llm.create_chat_completion(
            messages=[
                {"role": "system", "content": _SYSTEM_PROMPT},
                {"role": "user",   "content": prompt},
            ],
            max_tokens=512,
            temperature=1,
            top_p = 0.95,
            top_k = 64
        )
        raw = response["choices"][0]["message"]["content"].strip()
    except Exception as e:
        print(f"\n[Gemma] Inference error: {e}")
        return []

    if raw.upper().startswith("NONE") or not raw:
        return []

    goals = []
    for line in raw.splitlines():
        line = line.strip()
        if not line:
            continue
        # Strip leading numbering/bullets: "1.", "1)", "-", "*", "•"
        line = re.sub(r"^[\d]+[.)]\s*|^[-*•]\s*", "", line).strip()
        if len(line) > 5:
            goals.append(line)


    return goals


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def _parse_args():
    import argparse
    p = argparse.ArgumentParser(description="Extract learning goals from text using Gemma 4.")
    grp = p.add_mutually_exclusive_group(required=True)
    grp.add_argument("--text", metavar="TEXT", help="Chunk text (quoted string)")
    grp.add_argument("--file", metavar="FILE", help="Path to plain-text chunk file")
    p.add_argument("--repo-id",   default="unsloth/gemma-4-E4B-it-GGUF")
    p.add_argument("--filename",  default="gemma-4-E4B-it-Q4_K_M.gguf")
    p.add_argument("--n-ctx",     type=int, default=8192)
    p.add_argument("--max-goals", type=int, default=10)
    return p.parse_args()




In [ ]:
"""
chunk_blooms_tagger.py
======================
Classify text chunks (~1000 words) with Bloom's taxonomy levels and store
the result as metadata — suitable for RAG pipelines or vector store ingestion.

Model
-----
Fine-tuned BertForSequenceClassification (bert-base-uncased, 109M params).
  - Architecture : BERT-base → linear head (6 outputs, sigmoid, multilabel)
  - Trained on   : EDM2022CLO — short course learning outcome sentences
  - Checkpoint   : model/checkpoint-600  (step 600, ~2.8 epochs, eval_loss 0.0573)
  - Labels       : Remember · Understand · Apply · Analyze · Evaluate · Create

Why sentence-level decomposition?
----------------------------------
BERT has a hard 512-token limit (~380 words). A raw 1000-word chunk would be
silently truncated, losing ~60% of its content. Instead:
  1. Split each chunk into sentences using the same logic as the JD pipeline.
  2. Run all sentences in one flat BERT batch (efficient, no per-chunk loops).
  3. Aggregate: sum sigmoid probs across all sentences in a chunk, then pick
     the label with the highest summed score.
     (Same strategy as classify_explicit_blooms_v2 in the JD notebook.)

Metadata produced per chunk
---------------------------
  bloom_highest_level    : str        dominant Bloom's level (e.g. "Analyze")
  bloom_bucket           : str        coarse group: understand_bkt | analyze_bkt | create_bkt
  bloom_predicted_labels : list[str]  all labels above (scaled) threshold
  bloom_confidences      : dict       summed sigmoid scores per label
  bloom_sentence_count   : int        number of sentences used for inference
  understand_bkt_score   : float      Remember + Understand summed score
  analyze_bkt_score      : float      Apply + Analyze summed score
  create_bkt_score       : float      Evaluate + Create summed score

Usage — import
--------------
    from chunk_blooms_tagger import tag_chunks, load_blooms_model, tune_thresholds

    tokenizer, model, device = load_blooms_model("model/checkpoint-600")
    thresholds = tune_thresholds(DATA_URL, tokenizer, model, device)
    metadata = tag_chunks(chunks, tokenizer, model, device, thresholds)

Usage — CLI (single text test)
-------------------------------
    python chunk_blooms_tagger.py \\
        --model model/checkpoint-600 \\
        --text "Design a system that synthesizes user behavior data into insights."

Usage — CLI (JSON file of chunks)
----------------------------------
    python chunk_blooms_tagger.py \\
        --model model/checkpoint-600 \\
        --input chunks.json \\
        --output tagged_chunks.json

    chunks.json format: ["chunk text 1", "chunk text 2", ...]

Flags
-----
  --model       Path to checkpoint-600 directory (required)
  --text        Classify a single string; prints result to terminal
  --input       JSON file with list of chunk strings
  --output      Output JSON path (default: tagged_chunks.json, --input only)
  --thresholds  Path to a thresholds JSON file (optional; auto-detected otherwise)
  --data-url    URL/path to EDM2022CLO CSV — only needed the very first run
  --no-tune     Skip threshold tuning entirely; use flat 0.50 for all labels

Thresholds
----------
The per-label thresholds for checkpoint-600 are hardcoded as BLOOMS_TUNED_THRESHOLDS
and used by default — no tuning step, no network call, no extra file needed.
They were computed once from the notebook (Learning Goals_Blooms Level Inference.ipynb,
Step 5) with random_state=42 and are deterministic for this checkpoint:
    Remember: 0.50 | Understand: 0.45 | Apply: 0.55
    Analyze:  0.55 | Evaluate:   0.60 | Create: 0.60

Only use tune_thresholds() / --retune if you retrain the model on new data.
"""


from __future__ import annotations

import argparse
import json
import logging
import re
from pathlib import Path
from typing import Optional

import numpy as np

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

BLOOMS_LABEL_NAMES = ["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"]
BLOOMS_ORDER       = {label: idx for idx, label in enumerate(BLOOMS_LABEL_NAMES)}
BLOOMS_MAX_LENGTH  = 512
BLOOMS_BATCH_SIZE  = 32

# Pre-tuned thresholds for checkpoint-600 — computed once from the notebook
# (Learning Goals_Blooms Level Inference.ipynb, Step 5) with random_state=42.
# These are deterministic for this checkpoint and never need to be re-computed
# unless the model is retrained.
BLOOMS_TUNED_THRESHOLDS = {
    "Remember":   0.10,
    "Understand": 0.10,
    "Apply":      0.10,
    "Analyze":    0.10,
    "Evaluate":   0.10,
    "Create":     0.10,
}

# Only needed if you retrain and want to re-derive thresholds from scratch
DATA_URL            = "https://raw.githubusercontent.com/SteveLEEEEE/EDM2022CLO/refs/heads/main/data/sample_full.csv"
BLOOMS_RANDOM_STATE = 42

# ---------------------------------------------------------------------------
# Model loading
# ---------------------------------------------------------------------------

def load_blooms_model(model_path: str):
    """Load the fine-tuned BERT multilabel classifier."""
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    import os
    model_path = os.path.abspath(model_path)
    log.info(f"Loading Bloom's model from {model_path} on {device}...")

    # Load tokenizer from local checkpoint if saved there, else download once from HF.
    # Trained model weights always load locally (local_files_only=True).
    tok_source = model_path if (Path(model_path) / "vocab.txt").exists() else "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(tok_source)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, local_files_only=True
    ).to(device)
    model.eval()
    log.info(f"  Model loaded. Parameters: {model.num_parameters():,}")
    return tokenizer, model, device


def load_thresholds(path: str) -> dict | None:
    """Load cached thresholds from a JSON file. Returns None if file doesn't exist."""
    import os
    if not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as f:
        thresholds = json.load(f)
    log.info(f"  Loaded cached thresholds from {path}: {thresholds}")
    return thresholds


def tune_thresholds(data_source: str, tokenizer, model, device,
                    save_path: str | None = None) -> dict:
    """
    Find F1-maximising per-label threshold on the validation split.

    Results are deterministic (fixed model + fixed random seed), so pass
    save_path to cache them — subsequent runs will skip this entirely.
    Use load_thresholds(save_path) before calling this to avoid re-tuning.
    """
    import pandas as pd
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import f1_score

    log.info("Tuning Bloom's thresholds (one-time cost)...")
    data = pd.read_csv(data_source)
    data[BLOOMS_LABEL_NAMES] = data[BLOOMS_LABEL_NAMES].fillna(0).astype(np.float32)
    texts  = data["Learning_outcome"].tolist()
    labels = data[BLOOMS_LABEL_NAMES].values

    train_full, _, lbl_full, _ = train_test_split(
        texts, labels, test_size=0.2, random_state=BLOOMS_RANDOM_STATE
    )
    split = int(0.8 * len(lbl_full))
    val_texts  = train_full[split:]
    val_labels = lbl_full[split:]

    val_probs = _run_bert_batch(val_texts, tokenizer, model, device)

    sweep = np.arange(0.10, 0.91, 0.05)
    best_thresholds = {}
    for i, label in enumerate(BLOOMS_LABEL_NAMES):
        f1s = [
            f1_score(val_labels[:, i], (val_probs[:, i] > t).astype(int), zero_division=0)
            for t in sweep
        ]
        best_thresholds[label] = round(float(sweep[np.argmax(f1s)]), 4)

    log.info(f"  Tuned thresholds: {best_thresholds}")

    if save_path:
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(best_thresholds, f, indent=2)
        log.info(f"  Saved thresholds → {save_path}")

    return best_thresholds


# ---------------------------------------------------------------------------
# BERT inference
# ---------------------------------------------------------------------------

def _run_bert_batch(texts: list[str], tokenizer, model, device) -> np.ndarray:
    """Return sigmoid probabilities of shape (n, 6)."""
    import torch

    if not texts:
        return np.zeros((0, 6), dtype=np.float32)

    all_probs  = []
    n_batches  = (len(texts) + BLOOMS_BATCH_SIZE - 1) // BLOOMS_BATCH_SIZE
    log_every  = max(1, n_batches // 10)
    for batch_num, i in enumerate(range(0, len(texts), BLOOMS_BATCH_SIZE), 1):
        batch = texts[i : i + BLOOMS_BATCH_SIZE]
        encoded = tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=BLOOMS_MAX_LENGTH,
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            logits = model(**encoded).logits
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        if batch_num % log_every == 0 or batch_num == n_batches:
            sents_done = min(i + BLOOMS_BATCH_SIZE, len(texts))
            pct = sents_done / len(texts) * 100
            print(f"  [BERT] batch {batch_num}/{n_batches}  ({sents_done}/{len(texts)} sents, {pct:.0f}%)")
    return np.vstack(all_probs)


# ---------------------------------------------------------------------------
# Aggregation helpers
# ---------------------------------------------------------------------------

def _probs_to_labels(probs: np.ndarray, thresholds: dict) -> list[str]:
    predicted = [
        BLOOMS_LABEL_NAMES[j]
        for j, p in enumerate(probs)
        if p >= thresholds[BLOOMS_LABEL_NAMES[j]]
    ]
    return predicted if predicted else ["Below threshold"]


def _probs_to_bucket(probs: np.ndarray) -> dict:
    u = float(probs[0] + probs[1])
    a = float(probs[2] + probs[3])
    c = float(probs[4] + probs[5])
    winner = ["understand_bkt", "analyze_bkt", "create_bkt"][int(np.argmax([u, a, c]))]
    return {
        "bloom_bucket":           winner,
        "understand_bkt_score":   round(u, 4),
        "analyze_bkt_score":      round(a, 4),
        "create_bkt_score":       round(c, 4),
    }


# ---------------------------------------------------------------------------
# Sentence splitter (adapted from notebook)
# ---------------------------------------------------------------------------

_ABBREV_PROTECT = re.compile(
    r"\b(etc|e\.g|i\.e|vs|mr|mrs|dr|prof|sr|jr|dept|approx|"
    r"est|fig|no|vol|govt|pvt|ltd|inc|corp|st|ave|co|op)\.",
    re.IGNORECASE,
)
_BULLET_STRIP   = re.compile(r"^[\s]*(?:[•·\-\*]|(?:\d+|[a-zA-Z])[\.\)]\s*)\s*")
_MARKDOWN_STRIP = re.compile(r"\*{1,2}(.*?)\*{1,2}")


def _split_sentences(text: str) -> list[str]:
    """Split a chunk into clean sentences suitable for BERT input."""
    if not text or not isinstance(text, str):
        return []
    sentences = []
    for line in text.split("\n"):
        line = line.strip()
        if not line:
            continue
        clean_check = _MARKDOWN_STRIP.sub(r"\1", line).strip()
        if clean_check.endswith(":") or len(clean_check) < 5:
            continue
        protected = _ABBREV_PROTECT.sub(
            lambda m: m.group(0).replace(".", "\x00"), line
        )
        for part in re.split(r"\.\s+", protected):
            part = part.replace("\x00", ".")
            part = _BULLET_STRIP.sub("", part)
            part = _MARKDOWN_STRIP.sub(r"\1", part)
            part = part.rstrip(".").strip()
            if len(part) >= 5:
                sentences.append(part)
    return sentences


# ---------------------------------------------------------------------------
# Main public API
# ---------------------------------------------------------------------------

def tag_chunks(
    chunks: list[str],
    tokenizer,
    model,
    device,
    thresholds: Optional[dict] = None,
) -> list[dict]:
    """
    Classify a list of text chunks with Bloom's taxonomy levels.

    Parameters
    ----------
    chunks      : list of text strings (~1000 words each)
    tokenizer   : from load_blooms_model()
    model       : from load_blooms_model()
    device      : from load_blooms_model()
    thresholds  : per-label thresholds dict; defaults to BLOOMS_TUNED_THRESHOLDS
                  (pre-computed for checkpoint-600, no tuning needed)

    Returns
    -------
    List of metadata dicts, one per chunk:
        {
          "bloom_highest_level"   : str,        # e.g. "Analyze"
          "bloom_bucket"          : str,        # "understand_bkt" | "analyze_bkt" | "create_bkt"
          "bloom_predicted_labels": list[str],  # labels above threshold
          "bloom_confidences"     : dict,       # summed probs per label
          "bloom_sentence_count"  : int,        # sentences used for inference
          "understand_bkt_score"  : float,
          "analyze_bkt_score"     : float,
          "create_bkt_score"      : float,
        }
    """
    if thresholds is None:
        thresholds = BLOOMS_TUNED_THRESHOLDS

    # 1. Split every chunk into sentences; track which chunk each sentence belongs to
    all_sentences: list[str] = []
    chunk_boundaries: list[tuple[int, int]] = []  # (start_idx, end_idx) in all_sentences

    for chunk in chunks:
        sents = _split_sentences(chunk)
        start = len(all_sentences)
        all_sentences.extend(sents)
        chunk_boundaries.append((start, len(all_sentences)))

    log.info(f"  {len(chunks)} chunks → {len(all_sentences)} sentences for inference.")

    # 2. Single flat batch inference across all sentences
    if all_sentences:
        all_probs = _run_bert_batch(all_sentences, tokenizer, model, device)
    else:
        all_probs = np.zeros((0, 6), dtype=np.float32)

    # 3. Aggregate per chunk
    results: list[dict] = []
    for i, (start, end) in enumerate(chunk_boundaries):
        if start == end:
            # Chunk had no usable sentences
            results.append({
                "bloom_highest_level":    None,
                "bloom_bucket":           None,
                "bloom_predicted_labels": ["Below threshold"],
                "bloom_confidences":      {l: 0.0 for l in BLOOMS_LABEL_NAMES},
                "bloom_sentence_count":   0,
                "understand_bkt_score":   0.0,
                "analyze_bkt_score":      0.0,
                "create_bkt_score":       0.0,
            })
            continue

        chunk_probs = all_probs[start:end]          # (n_sents, 6)
        summed      = chunk_probs.sum(axis=0)       # (6,) — aggregate signal

        highest = BLOOMS_LABEL_NAMES[int(np.argmax(summed))]
        # Scale thresholds by number of sentences so multi-sentence chunks aren't
        # over-penalised (same logic as the explicit-cues aggregation in notebook)
        n_sents = end - start
        scaled_thresh = {l: thresholds[l] * n_sents for l in BLOOMS_LABEL_NAMES}
        predicted = [
            BLOOMS_LABEL_NAMES[j]
            for j, sp in enumerate(summed)
            if sp >= scaled_thresh[BLOOMS_LABEL_NAMES[j]]
        ] or ["Below threshold"]

        bucket_info = _probs_to_bucket(summed)

        results.append({
            "bloom_highest_level":    highest,
            "bloom_predicted_labels": predicted,
            "bloom_confidences":      {
                l: round(float(sp), 4)
                for l, sp in zip(BLOOMS_LABEL_NAMES, summed)
            },
            "bloom_sentence_count":   n_sents,
            **bucket_info,
        })

    return results


# ---------------------------------------------------------------------------
# Goal-based classifier (v2 — used with Gemma goal extractor)
# ---------------------------------------------------------------------------

def tag_chunks_from_goals(
    goals_per_chunk: list[list[str]],
    tokenizer,
    model,
    device,
    thresholds: Optional[dict] = None,
) -> list[dict]:
    """
    Classify chunks via pre-extracted learning goals instead of raw sentences.

    Parameters
    ----------
    goals_per_chunk : list of lists — one list of goal strings per chunk.
                      Goals should already be short (<= 512 tokens each).
    tokenizer / model / device : from load_blooms_model()
    thresholds : per-label thresholds applied directly to proportions (0-1).
                 Defaults to BLOOMS_TUNED_THRESHOLDS.

    Returns
    -------
    List of metadata dicts (same schema as tag_chunks), one per chunk.
    bloom_confidences values are proportions summing to ~1.0.
    Chunks with no goals get bloom_highest_level = None.
    """
    if thresholds is None:
        thresholds = BLOOMS_TUNED_THRESHOLDS

    # Flatten all goals and track per-chunk boundaries
    all_goals: list[str] = []
    boundaries: list[tuple[int, int]] = []
    for goals in goals_per_chunk:
        start = len(all_goals)
        all_goals.extend(goals)
        boundaries.append((start, len(all_goals)))

    # Single BERT batch over all goals
    if all_goals:
        all_probs = _run_bert_batch(all_goals, tokenizer, model, device)
    else:
        import numpy as _np
        all_probs = _np.zeros((0, 6), dtype=_np.float32)

    results: list[dict] = []
    for start, end in boundaries:
        n_goals = end - start

        if n_goals == 0:
            results.append({
                "bloom_highest_level":    "Remember",
                "bloom_bucket":           "understand_bkt",
                "bloom_predicted_labels": ["Remember"],
                "bloom_confidences":      {l: 0.0 for l in BLOOMS_LABEL_NAMES},
                "bloom_sentence_count":   0,
                "understand_bkt_score":   0.0,
                "analyze_bkt_score":      0.0,
                "create_bkt_score":       0.0,
            })
            continue

        chunk_probs = all_probs[start:end]          # (n_goals, 6)
        summed      = chunk_probs.sum(axis=0)       # (6,)
        total       = float(summed.sum()) or 1.0
        proportions = summed / total                # normalise → sum to 1.0

        highest  = BLOOMS_LABEL_NAMES[int(proportions.argmax())]
        predicted = [
            l for l, p in zip(BLOOMS_LABEL_NAMES, proportions)
            if p >= thresholds[l]
        ] or ["Below threshold"]

        bucket_info = _probs_to_bucket(proportions)

        results.append({
            "bloom_highest_level":    highest,
            "bloom_predicted_labels": predicted,
            "bloom_confidences":      {
                l: round(float(p), 4)
                for l, p in zip(BLOOMS_LABEL_NAMES, proportions)
            },
            "bloom_sentence_count":   n_goals,
            **bucket_info,
        })

    return results


# ---------------------------------------------------------------------------
# CLI entry point
# ---------------------------------------------------------------------------

def _parse_args():
    p = argparse.ArgumentParser(
        description="Tag text chunks with Bloom's levels.",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  # Test a quick string directly in the terminal
  python chunk_blooms_tagger.py --model model/checkpoint-600 --text "Design a system that can analyze user behavior and synthesize insights."

  # Run on a JSON file of chunks
  python chunk_blooms_tagger.py --model model/checkpoint-600 --input chunks.json --output tagged.json
        """,
    )
    p.add_argument("--model", required=True, help="Path to checkpoint-600 directory")
    p.add_argument(
        "--retune",
        action="store_true",
        help="Re-derive thresholds from the training data (only needed if you retrained "
             "the model). By default, hardcoded thresholds for checkpoint-600 are used.",
    )
    p.add_argument(
        "--data-url",
        default=DATA_URL,
        help="URL or local path to EDM2022CLO CSV — only used with --retune",
    )

    mode = p.add_mutually_exclusive_group(required=True)
    mode.add_argument("--text",  help="Classify a single text string directly from the terminal")
    mode.add_argument("--input", help="JSON file containing a list of chunk strings")

    p.add_argument("--output", default="tagged_chunks.json", help="Output JSON path (only used with --input)")
    return p.parse_args()


def _print_result(meta: dict, text_preview: str = "") -> None:
    """Pretty-print a single chunk's Bloom's result to the terminal."""
    bar_width = 30

    print("\n" + "=" * 60)
    if text_preview:
        print(f"  Text : {text_preview[:80]}{'...' if len(text_preview) > 80 else ''}")
    print(f"  Level : {meta['bloom_highest_level']}")
    print(f"  Bucket: {meta['bloom_bucket']}")
    print(f"  Labels: {', '.join(meta['bloom_predicted_labels'])}")
    print(f"  Sents : {meta['bloom_sentence_count']}")
    print()
    print("  Confidence breakdown (summed):")
    confs = meta["bloom_confidences"]
    max_val = max(confs.values()) if confs else 1.0
    for label in BLOOMS_LABEL_NAMES:
        val  = confs.get(label, 0.0)
        fill = int((val / max_val) * bar_width) if max_val > 0 else 0
        bar  = "█" * fill + "░" * (bar_width - fill)
        marker = " ◀" if label == meta["bloom_highest_level"] else ""
        print(f"    {label:<12} {bar} {val:.3f}{marker}")
    print("=" * 60)



In [ ]:
"""
Tag all RAG chunks in data/chunks.json with Bloom's Taxonomy metadata.

Pipeline (default):
  Phase 1 — Gemma 4 extracts learning goals/outcomes from each chunk.
  Phase 2 — BERT classifier tags each goal; proportions aggregated per chunk.

After running, execute:
    python scripts/export_for_web.py
to push bloom fields into webapp/data/chunks.json (no re-embedding).

Usage:
    python scripts/tag_blooms.py                  # tag only untagged chunks (Gemma + BERT)
    python scripts/tag_blooms.py --skip-gemma     # BERT only (sentence-split, original method)
    python scripts/tag_blooms.py --reset          # strip all bloom tags and re-tag from scratch
    python scripts/tag_blooms.py --rethreshold    # recompute bloom_predicted_labels from stored
                                                  # confidences using current BLOOMS_TUNED_THRESHOLDS
                                                  # (no model inference — instant)

Flags:
    --reset          Strip existing bloom fields from all chunks before tagging.
                     Use this after changing the model or retraining.

    --rethreshold    Recompute bloom_predicted_labels without re-running any model.
                     Edit BLOOMS_TUNED_THRESHOLDS in chunk_blooms_tagger.py, then
                     run this to instantly see how the label distribution changes.
                     bloom_highest_level is NOT affected (it is always argmax).

    --skip-gemma     Skip Gemma goal extraction; use original sentence-splitting
                     method. Faster but less accurate (BERT out-of-distribution).
"""

# Fix for notebook environment — __file__ is not defined in cells
from pathlib import Path
__file__ = "/kaggle/working/tag_blooms.py"

import sys
import json
import shutil
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parent.parent
_IS_KAGGLE   = Path('/kaggle/input').exists()

if _IS_KAGGLE:
    OFFLINE_DATA = Path('/kaggle/input/datasets/abdulkalamkhan/bloom-chunks-danny')
    MODEL_PATH   = Path('/kaggle/input/datasets/abdulkalamkhan/bloom-bert-classifier')
    OUTPUT_PATH  = Path('/kaggle/working/chunks_tagged.json')
else:
    OFFLINE_DATA = PROJECT_ROOT / "data"
    MODEL_PATH   = PROJECT_ROOT / "webapp" / "models" / "checkpoint-600"
    OUTPUT_PATH  = OFFLINE_DATA / "chunks.json"  # overwrites in-place locally

BLOOMS_ORDER = ["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"]



def _print_distribution(chunks):
    total     = len(chunks)
    bar_width = 30

    level_counts = Counter(c.get("bloom_highest_level") for c in chunks)
    print("\nBloom's highest level (one per chunk):")
    print(f"  {'Level':<12}  {'Count':>6}  {'%':>6}  Bar")
    for level in BLOOMS_ORDER:
        n   = level_counts.get(level, 0)
        pct = n / total * 100 if total else 0
        bar = "█" * int(pct / 100 * bar_width)
        print(f"  {level:<12}  {n:>6}  {pct:>5.1f}%  {bar}")
    unclassified = level_counts.get(None, 0)
    if unclassified:
        print(f"  {'(untagged)':<12}  {unclassified:>6}  {unclassified/total*100:>5.1f}%")

    label_counts = Counter()
    below_thresh = 0
    for c in chunks:
        labels = c.get("bloom_predicted_labels") or []
        if labels == ["Below threshold"]:
            below_thresh += 1
        else:
            label_counts.update(labels)
    print(f"\nBloom's predicted labels (chunks can appear in multiple):")
    print(f"  {'Level':<12}  {'Count':>6}  {'%':>6}  Bar")
    for level in BLOOMS_ORDER:
        n   = label_counts.get(level, 0)
        pct = n / total * 100 if total else 0
        bar = "█" * int(pct / 100 * bar_width)
        print(f"  {level:<12}  {n:>6}  {pct:>5.1f}%  {bar}")
    if below_thresh:
        print(f"  {'(no label)':<12}  {below_thresh:>6}  {below_thresh/total*100:>5.1f}%")


BLOOM_FIELDS = [
    "bloom_highest_level", "bloom_bucket", "bloom_predicted_labels",
    "bloom_confidences", "bloom_sentence_count",
    "understand_bkt_score", "analyze_bkt_score", "create_bkt_score",
]



def main():
    import argparse
    parser = argparse.ArgumentParser(description="Tag RAG chunks with Bloom's Taxonomy levels.")
    parser.add_argument("--reset",       action="store_true", help="Strip existing bloom tags and re-tag all chunks from scratch.")
    parser.add_argument("--rethreshold", action="store_true", help="Recompute bloom_predicted_labels from stored confidences using current thresholds (no model inference).")
    parser.add_argument("--skip-gemma",  action="store_true", help="Skip Gemma goal extraction; use original sentence-splitting method.")
    args = parser.parse_args()

    chunks_path = OFFLINE_DATA / "chunks.json"
    if not chunks_path.exists():
        print(f"Error: {chunks_path} not found.")
        sys.exit(1)
    print(f"{'[Kaggle] ' if _IS_KAGGLE else ''}Input : {chunks_path}")
    print(f"{'[Kaggle] ' if _IS_KAGGLE else ''}Output: {OUTPUT_PATH}")

    if not MODEL_PATH.exists():
        print(f"Error: model not found at {MODEL_PATH}")
        sys.exit(1)

    chunks = json.loads(chunks_path.read_text(encoding="utf-8"))
    print(f"Loaded {len(chunks)} chunks from data/chunks.json")

    # --rethreshold: recompute labels from stored proportions, no model needed
    if args.rethreshold:
        recomputed = 0
        for c in chunks:
            confs = c.get("bloom_confidences")
            if not confs:
                continue
            # proportions already stored (sum ~1.0) — apply thresholds directly
            predicted = [l for l in BLOOMS_LABEL_NAMES if confs.get(l, 0) >= BLOOMS_TUNED_THRESHOLDS[l]] or ["Below threshold"]
            c["bloom_predicted_labels"] = predicted
            recomputed += 1
        OUTPUT_PATH.write_text(json.dumps(chunks, indent=2), encoding="utf-8")
        print(f"Recomputed bloom_predicted_labels for {recomputed} chunks using current thresholds.")
        _print_distribution(chunks)
        return

    if args.reset:
        for c in chunks:
            for f in BLOOM_FIELDS:
                c.pop(f, None)
        print(f"Reset: stripped bloom fields from all {len(chunks)} chunks.")

    untagged_idx = [
        i for i, c in enumerate(chunks)
        if not c.get("bloom_highest_level")
    ]
    print(f"Already tagged : {len(chunks) - len(untagged_idx)}")
    print(f"To tag         : {len(untagged_idx)}")

    if not untagged_idx:
        print("All chunks already tagged — nothing to do.")
        _print_distribution(chunks)
        return

    texts = [chunks[i]["content"] for i in untagged_idx]

    if args.skip_gemma:
        # Original method: sentence splitting → BERT
        print(f"\nLoading BERT from {MODEL_PATH} ...")
        tok, bert_model, device = load_blooms_model(str(MODEL_PATH))
        print(f"Model on {device}")
        metas = tag_chunks(texts, tok, bert_model, device)

    else:
        llm = load_gemma()
        goals_per_chunk = []
        log_every = max(1, len(texts) // 20)  # ~20 log lines total
        for idx, text in enumerate(texts):
            goals = extract_goals(text, llm)
            goals_per_chunk.append(goals)
            if (idx + 1) % log_every == 0 or idx == len(texts) - 1:
                total_goals = sum(len(g) for g in goals_per_chunk)
                print(f"  [Gemma] {idx+1}/{len(texts)} chunks  ({total_goals} goals so far)")
        print(f"[Gemma] Done. Total goals extracted: {sum(len(g) for g in goals_per_chunk)}")
        zero_goal = sum(1 for g in goals_per_chunk if not g)
        if zero_goal:
            print(f"[Gemma] Warning: {zero_goal} chunk(s) returned no goals — will be marked unclassified.")

        # Phase 2: BERT classification on extracted goals
        print(f"\nLoading BERT from {MODEL_PATH} ...")
        tok, bert_model, device = load_blooms_model(str(MODEL_PATH))
        print(f"Model on {device}")
        metas = tag_chunks_from_goals(goals_per_chunk, tok, bert_model, device)

    for i, meta in zip(untagged_idx, metas):
        chunks[i].update(meta)

    if not _IS_KAGGLE:
        bak = OUTPUT_PATH.with_suffix(".json.bak")
        shutil.copy2(OUTPUT_PATH, bak)
        print(f"\nBacked up → {bak.name}")

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_PATH.write_text(json.dumps(chunks, indent=2), encoding="utf-8")
    print(f"Saved {len(chunks)} chunks → {OUTPUT_PATH}")

    _print_distribution(chunks)

    print(f"\nDone. Bloom tags written for {len(untagged_idx)} chunks.")
    print("Next: python scripts/export_for_web.py")



In [ ]:
import sys
sys.argv = ["tag_blooms.py", "--reset"]

# Override load_gemma to respect the detected GPU setting
_orig_load_gemma = load_gemma
def load_gemma(**kwargs):
    kwargs.setdefault("n_gpu_layers", N_GPU_LAYERS)
    return _orig_load_gemma(**kwargs)

main()

In [ ]:
# Cell 6 — Verify output
import json
with open('/kaggle/working/chunks_tagged.json') as f:
    chunks = json.load(f)
tagged = sum(1 for c in chunks if c.get('bloom_highest_level'))
print(f"{tagged}/{len(chunks)} chunks tagged")
print("Download chunks_tagged.json from the output panel on the right.")